In [2]:
# =============================================================================
# PART 1 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 0,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %% [markdown]
# # Inventory Rebalancing and Demand Forecasting Pipeline
#
# End-to-end pipeline: demand forecasting, hierarchical forecast reconciliation,
# and interdepot inventory transfer recommendations.
#
# Run cell-by-cell in VS Code (Shift+Enter), or top-to-bottom as a script.
#
# **Outputs produced (saved to the `output/` folder):**
# - `transfer_recommendations.csv` — every recommended transfer: SKU, source
#   store, destination store, quantity (cartons and units), urgency and
#   preferred transfer date
# - `value_impact.csv` — estimated value impact of each recommended transfer
# - `network_impact_summary.csv` — one-line network-wide impact summary
# - `sku_categorisation.csv` — SKU-level categorisation and priority ranking
# - `momentum_signals.csv` — short-term demand momentum by store/SKU

# %%
# =============================================================================
# 1. CONFIGURATION
# =============================================================================
#
# Required packages:
#   pip install pandas numpy pyarrow xgboost scipy scikit-learn prophet

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import math
import logging

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import xgboost as xgb
except ImportError:
    sys.exit("ERROR: xgboost is not installed. Run:  pip install xgboost")

# ---- Paths ------------------------------------------------------------------
FILE_PATH  = r"C:\Users\dpdea\Desktop\Cranfield\Thesis\master_egitim_df2.parquet"  # ← change this
OUTPUT_DIR = os.path.join(os.path.dirname(FILE_PATH), "output")  # created next to the data file

if not os.path.exists(FILE_PATH):
    sys.exit(
        f"ERROR: File not found at:\n  {FILE_PATH}\n"
        f"Update FILE_PATH at the top of this script before running."
    )
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR}")

with open(os.path.join(OUTPUT_DIR, "README_IMPORTANT.txt"), "w") as f:
    f.write(
        "IMPORTANT -- READ BEFORE USING THESE FILES\n"
        + ("=" * 60) + "\n\n"
        "All figures labelled 'estimated' in these files are PREDICTIONS,\n"
        "not actual, confirmed, or measured values. Specifically:\n\n"
        "- No fixed price list or cost data was available in the source data.\n"
        "  Unit prices are inferred from historical revenue divided by units\n"
        "  sold for each product, which is an approximation, not a real price.\n\n"
        "- Cost figures assume a generic 60% cost-of-goods-sold ratio applied\n"
        "  to that estimated price. This is a placeholder assumption, not\n"
        "  derived from actual supplier or accounting cost data.\n\n"
        "- Demand forecasts are model predictions based on historical sales\n"
        "  patterns and are subject to forecast error.\n\n"
        "These figures are intended to help PRIORITISE which transfers matter\n"
        "most (relative comparison), not to be booked or reported as actual\n"
        "revenue, cost, or profit. If actual price and cost data becomes\n"
        "available, these estimates should be replaced or validated against it\n"
        "before being used in any financial reporting or decision with\n"
        "financial consequences.\n\n"
        "File-by-file summary:\n"
        "- transfer_recommendations.csv : recommended transfers (predicted demand)\n"
        "- unmatched_shortages.csv      : shortages with no available donor store\n"
        "- unused_donor_capacity.csv    : surplus stock not currently needed elsewhere\n"
        "- inventory_status_report.csv  : full inventory status, all products/stores\n"
        "- value_impact.csv             : estimated value impact per transfer\n"
        "- network_impact_summary.csv   : estimated network-wide impact summary\n"
        "- sku_categorisation.csv       : product priority ranking\n"
        "- momentum_signals.csv         : short-term demand trend signals\n"
        "- backtest_results.csv         : validation check comparing recipients' actual\n"
        "                                 sales before vs after a historical cutoff date;\n"
        "                                 see Section 16 in the script for full detail\n"
        "                                 and important caveats\n"
    )

# ---- Data loading -------------------------------------------------------------
CHUNK_ROWS      = 500_000
FORECAST_SAMPLE = 0.70   # sample used to train the forecasting/reconciliation models
                          # (the transfer-recommendation logic below uses 100% of the data)

# ---- Inventory / transfer parameters ------------------------------------------
EXCESS_THRESHOLD      = 1.40   # demand > 140% of network median (see Above-Network Demand documentation in Section 9)
SHORTAGE_THRESHOLD    = 0.65   # demand < 65% of network median  -> Shortage Risk
CRITICAL_STOCKOUT_PCT = 10     # stockout rate >= 10%             -> Critical Shortage
UNITS_PER_CARTON      = 12
TRANSFER_HORIZON_DAYS = 14
DONOR_KEEP_BACK_RATIO = 0.20   # a donor keeps back 20% of its calculated surplus as its own
                                # buffer, rather than being drawn all the way down to the
                                # network median with nothing held in reserve
ASSUMED_COGS_RATIO    = 0.60   # generic assumption -- update if actual cost data is available

REQUIRED_COLS = ["tarih", "magazakodu", "urunkodu", "satismiktari", "satistutarikdvsiz",
                  "promotion_day", "season_code", "stok_out"]

_schema_cols = set(pq.ParquetFile(FILE_PATH).schema_arrow.names)
_missing_cols = [c for c in REQUIRED_COLS if c not in _schema_cols]
if _missing_cols:
    sys.exit(
        f"ERROR: Input file is missing required column(s): {_missing_cols}\n"
        f"Columns found: {sorted(_schema_cols)}"
    )

READ_COLS = REQUIRED_COLS
print("Configuration loaded.")
print("=" * 70)
print("IMPORTANT: This pipeline produces PREDICTIONS AND ESTIMATES only.")
print("No actual price list or cost data was available in the source data.")
print("See README_IMPORTANT.txt in the output folder for full details.")
print("=" * 70)

Output folder: C:\Users\dpdea\Desktop\Cranfield\Thesis\output
Configuration loaded.
IMPORTANT: This pipeline produces PREDICTIONS AND ESTIMATES only.
No actual price list or cost data was available in the source data.
See README_IMPORTANT.txt in the output folder for full details.


In [3]:
# =============================================================================
# PART 2 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 1,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 2. DATA LOADING
# =============================================================================

def load_data(file_path: str, columns: list, chunk_rows: int = CHUNK_ROWS) -> pd.DataFrame:
    """Reads the Parquet file in batches to keep memory usage bounded, then
    returns the full combined dataset (no rows are skipped or sampled)."""
    pf = pq.ParquetFile(file_path)
    chunks = []
    for batch in pf.iter_batches(batch_size=chunk_rows, columns=columns):
        tbl = pa.Table.from_batches([batch])
        new_cols = {}
        for i, field in enumerate(tbl.schema):
            col = tbl.column(i)
            if pa.types.is_float64(field.type):
                col = col.cast(pa.float32())
            elif pa.types.is_int64(field.type):
                col = col.cast(pa.int32())
            new_cols[field.name] = col
        chunks.append(pa.table(new_cols).to_pandas(self_destruct=True))
        del tbl, batch
    out = pd.concat(chunks, ignore_index=True)
    del chunks
    return out


print("Loading data...")
df_full = load_data(FILE_PATH, READ_COLS)

df_full["tarih"]         = pd.to_datetime(df_full["tarih"])
df_full["magazakodu"]    = df_full["magazakodu"].astype("category")
df_full["urunkodu"]      = df_full["urunkodu"].astype("category")
df_full["promotion_day"] = df_full["promotion_day"].fillna(0).astype("int8")
df_full["season_code"]   = df_full["season_code"].fillna(0).astype("int8")
df_full["stok_out"]      = df_full["stok_out"].fillna(0).astype("int8")
df_full["month"]         = df_full["tarih"].dt.month.astype("int8")

def get_season(m):
    if m in (12, 1, 2):  return "Winter"
    if m in (3, 4, 5):   return "Spring"
    if m in (6, 7, 8):   return "Summer"
    return "Autumn"

df_full["season"] = df_full["month"].apply(get_season).astype("category")
df_full = df_full.sort_values(["magazakodu", "urunkodu", "tarih"]).reset_index(drop=True)

print(f"  Loaded {len(df_full):,} rows | "
      f"{df_full['magazakodu'].nunique()} stores | {df_full['urunkodu'].nunique():,} SKUs")

Loading data...
  Loaded 25,746,847 rows | 33 stores | 743 SKUs


In [4]:
# =============================================================================
# PART 3 OF 16 -- Inventory Rebalancing Pipeline
# Extract from the master file (Inventory_Rebalancing_Pipeline.py) for readability
# only. Depends on variables from Parts 1 through 2. Not standalone-runnable.
# =============================================================================

# %%
# =============================================================================
# 3. FEATURE ENGINEERING
# =============================================================================
# Two separate EMA (exponential moving average) demand signals are computed:
#   - Revenue-based EMA: used to train the demand forecasting model
#   - Units-based EMA: used for inventory status classification and transfer
#     decisions, since transfers move physical stock, not currency

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    IMPORTANT: sales_roll_mean_7 and the ema_* columns below are computed on
    the series SHIFTED BY ONE DAY FIRST (grp.shift(1) before rolling/ewm), not
    on the raw series directly. Without that shift, pandas' rolling() and
    ewm() windows both include the CURRENT row's own value by default, which
    means a feature for day T would partially contain day T's own target
    value (satistutarikdvsiz) -- the model would be predicting a value using
    a feature that already leaks a smoothed version of that same value. This
    is a same-row leakage issue, separate from and in addition to the
    train/test split leakage fixed in Section 4; it exists on every row
    regardless of which side of that split the row falls on. Shifting first
    means day T's features reflect only data through day T-1, consistent
    with how sales_lag_1 already behaved before this fix.
    """
    data = data.copy()
    data["year"]       = data["tarih"].dt.year.astype("int16")
    data["weekday"]    = data["tarih"].dt.weekday.astype("int8")
    data["is_weekend"] = (data["weekday"] >= 5).astype("int8")

    grp = data.groupby(["magazakodu", "urunkodu"], observed=True)["satistutarikdvsiz"]
    shifted = grp.shift(1)   # yesterday and earlier only -- the anchor for every feature below
    shifted_grp = shifted.groupby([data["magazakodu"], data["urunkodu"]], observed=True)

    data["sales_lag_1"]       = shifted
    data["sales_roll_mean_7"] = shifted_grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
    for span in (7, 14, 30, 90):
        data[f"ema_{span}"] = shifted_grp.transform(lambda x: x.ewm(span=span, adjust=False).mean()).astype("float32")
    data["ema_trend"] = (data["ema_7"] - data["ema_90"]).astype("float32")
    data["ema_ratio"] = (data["ema_7"] / data["ema_90"].replace(0, np.nan)).astype("float32")
    return data

df_full = engineer_features(df_full)

# NOTE: the units-based EMA below (ema_*_u) is deliberately NOT shifted, unlike
# the revenue-based features above. It is used only as a live "current smoothed
# demand" signal for the transfer/status logic in Sections 7-13 -- it is never
# evaluated against a held-out prediction target the way the forecasting
# features are, so including today's own value is correct here: when deciding
# right now whether a store needs a transfer, you want today's most recent
# data reflected in the signal, not artificially withheld from it.
_units_grp = df_full.groupby(["magazakodu", "urunkodu"], observed=True)["satismiktari"]
for span in (7, 14, 30, 90):
    df_full[f"ema_{span}_u"] = _units_grp.transform(lambda x: x.ewm(span=span, adjust=False).mean()).astype("float32")
df_full["ema_trend_u"] = (df_full["ema_7_u"] - df_full["ema_90_u"]).astype("float32")
df_full["ema_ratio_u"] = (df_full["ema_7_u"] / df_full["ema_90_u"].replace(0, np.nan)).astype("float32")

print("Feature engineering complete.")

Feature engineering complete.


In [5]:
# =============================================================================
# PART 4 OF 16 -- Inventory Rebalancing Pipeline
# Extract from the master file (Inventory_Rebalancing_Pipeline.py) for readability
# only. Depends on variables from Parts 1 through 3. Not standalone-runnable.
# =============================================================================

# %%
# =============================================================================
# 4. DEMAND FORECASTING MODEL
# =============================================================================
# Uses a CHRONOLOGICAL split (last 20% of dates held out), not a random split.
# A random split on lagged/EMA features leaks future information into
# training, since sales_lag_1 and the EMA columns are computed using the full
# timeline before any split happens -- a randomly selected "test" row can sit
# chronologically before rows used to train on. The cutoff below guarantees
# every test row occurs strictly after every training row.

df_fc = df_full.sample(frac=FORECAST_SAMPLE, random_state=42).sort_values(
    ["magazakodu", "urunkodu", "tarih"]
).reset_index(drop=True)
df_fc = df_fc.dropna(subset=["sales_lag_1", "ema_90"]).reset_index(drop=True)

FORECAST_FEATURES = ["year", "month", "weekday", "is_weekend", "promotion_day", "season_code",
                      "sales_lag_1", "sales_roll_mean_7", "stok_out",
                      "ema_7", "ema_30", "ema_90", "ema_trend", "ema_ratio"]

cutoff_date = df_fc["tarih"].quantile(0.8)
train_mask = df_fc["tarih"] < cutoff_date
test_mask  = df_fc["tarih"] >= cutoff_date

X_train, y_train = df_fc.loc[train_mask, FORECAST_FEATURES], df_fc.loc[train_mask, "satistutarikdvsiz"]
X_test,  y_test  = df_fc.loc[test_mask,  FORECAST_FEATURES], df_fc.loc[test_mask,  "satistutarikdvsiz"]
idx_test = df_fc.index[test_mask]

print(f"Chronological split: train ends {df_fc.loc[train_mask, 'tarih'].max().date()}, "
      f"test starts {df_fc.loc[test_mask, 'tarih'].min().date()} (cutoff at 80th percentile of dates)")

bottom_model = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.15, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", random_state=42, eval_metric="mae",
)
bottom_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

df_fc.loc[idx_test, "forecast"] = bottom_model.predict(X_test)
test_df = df_fc.loc[idx_test].copy()
forecast_mae = mean_absolute_error(test_df["satistutarikdvsiz"], test_df["forecast"])

# Naive baseline: "tomorrow will look like the last 7-day average" (sales_roll_mean_7).
# If the trained model can't beat this, it isn't adding value over a trivial rule.
naive_mae = mean_absolute_error(y_test, X_test["sales_roll_mean_7"])
print(f"Demand forecasting model | Test MAE: {forecast_mae:.2f}")
print(f"Naive baseline (7-day rolling average) | Test MAE: {naive_mae:.2f}")
if forecast_mae < naive_mae:
    print(f"Model beats the naive baseline by {(naive_mae - forecast_mae) / naive_mae * 100:.1f}%")
else:
    print("WARNING: model does not beat the naive baseline -- forecast is not adding value.")

Chronological split: train ends 2025-06-06, test starts 2025-06-07 (cutoff at 80th percentile of dates)
Demand forecasting model | Test MAE: 120.15
Naive baseline (7-day rolling average) | Test MAE: 116.03


In [6]:
# =============================================================================
# PART 5 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 4,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 5. HIERARCHY CONSTRUCTION (Store x SKU -> Store -> Network total)
# =============================================================================

bottom_series = (
    test_df.groupby(["tarih", "magazakodu", "urunkodu"], observed=True)
    .agg(actual=("satistutarikdvsiz", "sum"), forecast=("forecast", "sum"))
    .reset_index()
)
bottom_series["series_id"] = (
    bottom_series["magazakodu"].astype(str) + "_" + bottom_series["urunkodu"].astype(str)
)

actual_wide   = bottom_series.pivot_table(index="tarih", columns="series_id", values="actual",   aggfunc="sum", fill_value=0)
forecast_wide = bottom_series.pivot_table(index="tarih", columns="series_id", values="forecast", aggfunc="sum", fill_value=0)

bottom_cols = list(actual_wide.columns)
n_bottom    = len(bottom_cols)

series_to_store = bottom_series.drop_duplicates("series_id").set_index("series_id")["magazakodu"]
stores  = series_to_store.unique()
n_store = len(stores)
n_total = 1 + n_store + n_bottom

rows_, cols_, data_ = [], [], []
for j in range(n_bottom):
    rows_.append(0); cols_.append(j); data_.append(1.0)
series_store_arr = series_to_store.loc[bottom_cols].values
for i, store in enumerate(stores):
    for j in np.where(series_store_arr == store)[0]:
        rows_.append(1 + i); cols_.append(j); data_.append(1.0)
for j in range(n_bottom):
    rows_.append(1 + n_store + j); cols_.append(j); data_.append(1.0)

S = sparse.csr_matrix((data_, (rows_, cols_)), shape=(n_total, n_bottom))

base_forecasts = forecast_wide[bottom_cols].values
actuals_bottom = actual_wide[bottom_cols].values
y_hat_all  = base_forecasts @ S.T
y_all      = actuals_bottom @ S.T
date_index = actual_wide.index

print(f"Hierarchy built: {n_bottom} store-SKU combinations, {n_store} stores, 1 network total")

Hierarchy built: 23063 store-SKU combinations, 33 stores, 1 network total


In [7]:
# =============================================================================
# PART 6 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 5,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 6. FORECAST RECONCILIATION
# =============================================================================
# Ensures forecasts are consistent across levels (SKU totals sum correctly to
# store totals, and store totals sum correctly to the network total).

store_to_idx = {s: i for i, s in enumerate(stores)}
series_to_store_idx = np.array([store_to_idx[s] for s in series_store_arr])


def reconcile(S, y_hat_all, w_diag, series_to_store_idx, n_store):
    w_inv = 1.0 / w_diag
    a = w_inv[0]
    b = w_inv[1:1 + n_store]
    d = np.where(w_inv[1 + n_store:] <= 0, 1e-12, w_inv[1 + n_store:])
    d_inv = 1.0 / d

    sum_dinv_total     = d_inv.sum()
    sum_dinv_per_store = np.bincount(series_to_store_idx, weights=d_inv, minlength=n_store)

    K = 1 + n_store
    UtDinvU = np.zeros((K, K))
    UtDinvU[0, 0]  = sum_dinv_total
    UtDinvU[0, 1:] = sum_dinv_per_store
    UtDinvU[1:, 0] = sum_dinv_per_store
    UtDinvU[1:, 1:] = np.diag(sum_dinv_per_store)

    C_inv = np.diag(1.0 / np.concatenate([[a], b]))
    inner = np.linalg.inv(C_inv + UtDinvU)

    def apply_inverse(v):
        Dinv_v = v * d_inv
        total_sum  = Dinv_v.sum(axis=1)
        store_sums = np.zeros((v.shape[0], n_store))
        for s in range(n_store):
            mask = series_to_store_idx == s
            store_sums[:, s] = Dinv_v[:, mask].sum(axis=1)
        Ut_Dinv_v  = np.column_stack([total_sum, store_sums])
        correction = Ut_Dinv_v @ inner.T
        U_correction = correction[:, 0:1] + correction[:, 1:][:, series_to_store_idx]
        return Dinv_v - d_inv[None, :] * U_correction

    StWinv      = S.T.multiply(w_inv)
    StWinv_yhat = y_hat_all @ StWinv.T.toarray()
    y_tilde_bottom = apply_inverse(StWinv_yhat)
    return y_tilde_bottom @ S.T.toarray()


w_ols = np.ones(n_total)
y_ols = reconcile(S, y_hat_all, w_ols, series_to_store_idx, n_store)

# WLS: weight each node by its own out-of-sample residual variance, computed
# from the chronological held-out period above -- not uniform weights, and
# not evaluated on the training window the forecast was fit on.
residuals = y_all - y_hat_all
var_diag = np.var(residuals, axis=0)
var_diag[var_diag == 0] = var_diag[var_diag > 0].mean() if (var_diag > 0).any() else 1.0
y_wls = reconcile(S, y_hat_all, var_diag, series_to_store_idx, n_store)

# MinT-shrinkage: shrink each node's variance toward the pooled (grand-mean)
# variance, controlled by a data-driven shrinkage intensity lambda.
grand_mean_var  = var_diag.mean()
between_var     = np.var(var_diag)
shrink_lambda   = np.clip(grand_mean_var / (grand_mean_var + between_var) if (grand_mean_var + between_var) > 0 else 1.0, 0, 1)
w_mint          = shrink_lambda * grand_mean_var + (1 - shrink_lambda) * var_diag
w_mint[w_mint <= 0] = grand_mean_var
y_mint = reconcile(S, y_hat_all, w_mint, series_to_store_idx, n_store)

reconciliation_results = {}
for name, y_pred in [("OLS", y_ols), ("WLS", y_wls), (f"MinT (shrinkage lambda={shrink_lambda:.3f})", y_mint)]:
    mae  = mean_absolute_error(y_all.ravel(), y_pred.ravel())
    rmse = np.sqrt(mean_squared_error(y_all.ravel(), y_pred.ravel()))
    reconciliation_results[name] = (mae, rmse)
    print(f"Reconciled forecast [{name}] | MAE: {mae:.2f} | RMSE: {rmse:.2f}")

# Use WLS as the default reconciled forecast passed to the rest of the pipeline,
# since it is genuinely weighted rather than uniform and is the more defensible
# default of the three where MinT's shrinkage estimate is close to zero.
#
# NOTE: y_reconciled is computed and reported here but is NOT currently
# consumed by the transfer allocation logic in Section 10 -- that logic reads
# demand from df_full's units-based EMA signal instead. This is the
# still-open item from the code review (the forecast model is built and
# reconciled but not yet wired into allocation). Section 9's profile table is
# the integration point once that is done: replace its avg_demand column with
# a per-store-SKU forecast derived from y_reconciled.
y_reconciled = y_wls
recon_mae, recon_rmse = reconciliation_results["WLS"]

Reconciled forecast [OLS] | MAE: 151.86 | RMSE: 13498.40
Reconciled forecast [WLS] | MAE: 151.86 | RMSE: 13498.40
Reconciled forecast [MinT (shrinkage lambda=0.000)] | MAE: 151.86 | RMSE: 13498.40


In [8]:
# =============================================================================
# PART 7 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 6,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 7. DEMAND MOMENTUM SIGNAL (short-term trend, by store and SKU)
# =============================================================================

def momentum_label(ratio):
    if pd.isna(ratio):  return "Unknown"
    if ratio > 1.10:     return "Accelerating"
    if ratio > 1.02:     return "Rising"
    if ratio >= 0.98:    return "Stable"
    if ratio >= 0.90:    return "Falling"
    return "Decelerating"

momentum = (
    df_full.groupby(["magazakodu", "urunkodu"], observed=True)
    .agg(latest_ema_7=("ema_7_u", "last"), latest_ema_14=("ema_14_u", "last"),
         total_units_sold=("satismiktari", "sum"))
    .reset_index()
)
momentum["momentum_ratio"] = momentum["latest_ema_7"] / momentum["latest_ema_14"].replace(0, np.nan)
momentum["momentum_signal"] = momentum["momentum_ratio"].apply(momentum_label)

momentum_out = momentum.rename(columns={"magazakodu": "store", "urunkodu": "sku"})
momentum_out.to_csv(os.path.join(OUTPUT_DIR, "momentum_signals.csv"), index=False)

print("Demand momentum summary:")
print(momentum["momentum_signal"].value_counts().to_string())
print(f"Saved: {OUTPUT_DIR}/momentum_signals.csv")

Demand momentum summary:
momentum_signal
Decelerating    17735
Falling          2151
Rising           1040
Accelerating      816
Stable            800
Unknown           521
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/momentum_signals.csv


In [9]:
# =============================================================================
# PART 8 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 7,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 8. SKU CATEGORISATION AND TRANSFER PRIORITY
# =============================================================================

sku_profile = (
    df_full.groupby("urunkodu", observed=True)
    .agg(avg_daily_sales=("satismiktari", "mean"), std_daily_sales=("satismiktari", "std"),
         stockout_rate_pct=("stok_out", "mean"), promotion_rate_pct=("promotion_day", "mean"),
         avg_demand_trend=("ema_trend_u", "mean"), stores_selling=("magazakodu", "nunique"))
    .reset_index()
)
sku_profile["stockout_rate_pct"]  *= 100
sku_profile["promotion_rate_pct"] *= 100
sku_profile["demand_volatility"] = sku_profile["std_daily_sales"] / sku_profile["avg_daily_sales"].replace(0, np.nan)

p70, p30 = sku_profile["avg_daily_sales"].quantile([0.70, 0.30])

def categorise(row):
    cats = []
    if row["stockout_rate_pct"] >= 10:     cats.append("Stockout-Prone")
    if row["avg_daily_sales"] >= p70:       cats.append("Fast-Moving")
    elif row["avg_daily_sales"] <= p30:     cats.append("Slow-Moving")
    if row["avg_demand_trend"] < -2:        cats.append("Declining")
    if row["promotion_rate_pct"] >= 20:     cats.append("Promotion-Driven")
    if row["demand_volatility"] > 1.5:      cats.append("High-Volatility")
    return "|".join(cats) if cats else "Standard"

def priority(row):
    cats = row["category"]
    if "Stockout-Prone" in cats and "Fast-Moving" in cats: return 1
    if "Stockout-Prone" in cats:                            return 2
    if "Fast-Moving" in cats:                               return 3
    if "Declining" in cats or "Slow-Moving" in cats:        return 6
    return 5

sku_profile["category"]          = sku_profile.apply(categorise, axis=1)
sku_profile["transfer_priority"] = sku_profile.apply(priority, axis=1)

sku_profile.rename(columns={"urunkodu": "sku"}).sort_values("transfer_priority").to_csv(
    os.path.join(OUTPUT_DIR, "sku_categorisation.csv"), index=False
)

print("SKU transfer priority distribution (1 = highest priority):")
print(sku_profile["transfer_priority"].value_counts().sort_index().to_string())
print(f"Saved: {OUTPUT_DIR}/sku_categorisation.csv")

SKU transfer priority distribution (1 = highest priority):
transfer_priority
1     39
2    196
3    184
5    201
6    123
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/sku_categorisation.csv


In [10]:
# =============================================================================
# PART 9 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 8,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 9. INVENTORY STATUS CLASSIFICATION
# =============================================================================

df_full["implied_unit_value"] = np.where(
    df_full["satismiktari"] > 0, df_full["satistutarikdvsiz"] / df_full["satismiktari"], np.nan
).astype("float32")

sku_value = (
    df_full[df_full["implied_unit_value"] > 0]
    .groupby("urunkodu", observed=True)["implied_unit_value"]
    .median().reset_index().rename(columns={"implied_unit_value": "median_unit_value"})
)

# -----------------------------------------------------------------------------
# IMPORTANT LIMITATION -- READ BEFORE INTERPRETING "Above-Network Demand"
# -----------------------------------------------------------------------------
# The source data contains no stock-on-hand or inventory-position field of any
# kind (see REQUIRED_COLS in Section 1). "Above-Network Demand" therefore does
# NOT mean a store has excess physical inventory sitting in the back room --
# it means the store SELLS more of a product than a typical store, which is a
# demand-side observation, not an inventory-side one. A store can rank as
# Above-Network Demand while still being tightly stocked relative to its own
# sales pace, or even under-supplied relative to its own demand.
#
# This status is used as a DONOR CANDIDATE signal, not a confirmed surplus
# signal. Before executing any transfer recommended on this basis, the
# donor store's actual stock-on-hand should be checked. If a stock-on-hand
# feed becomes available, replace avg_demand-based classification below with
# a stock-cover based one (e.g. stock-on-hand / forecast demand).
# -----------------------------------------------------------------------------

profile = (
    df_full.groupby(["magazakodu", "urunkodu", "season"], observed=True)
    .agg(avg_demand=("ema_30_u", "mean"), avg_daily_sales=("satismiktari", "mean"),
         avg_daily_rev=("satistutarikdvsiz", "mean"), days_recorded=("tarih", "count"),
         stockout_days=("stok_out", "sum"))
    .reset_index()
)
profile["stockout_rate_pct"] = (profile["stockout_days"] / profile["days_recorded"] * 100).round(2)
profile = profile.merge(sku_value, on="urunkodu", how="left")

benchmark = (
    df_full.groupby(["urunkodu", "season"], observed=True)["ema_30_u"]
    .median().reset_index().rename(columns={"ema_30_u": "network_median_demand"})
)
profile = profile.merge(benchmark, on=["urunkodu", "season"], how="left")
profile["demand_vs_network"] = (profile["avg_demand"] / profile["network_median_demand"].replace(0, np.nan)).round(3)

def classify_status(row):
    """
    Critical Shortage : stockout rate >= threshold, checked first and
                         independently of demand level.
    Shortage Risk      : demand below threshold relative to network median.
    Above-Network Demand : demand above threshold relative to network median.
                         NOT a confirmed inventory surplus -- see limitation
                         note above.
    Normal             : none of the above.
    """
    if row["stockout_rate_pct"] >= CRITICAL_STOCKOUT_PCT: return "Critical Shortage"
    if row["demand_vs_network"] < SHORTAGE_THRESHOLD:     return "Shortage Risk"
    if row["demand_vs_network"] > EXCESS_THRESHOLD:       return "Above-Network Demand"
    return "Normal"

profile["status"] = profile.apply(classify_status, axis=1)

print("Inventory status classification (store x SKU x season):")
print(profile["status"].value_counts().to_string())

# Full status report -- every store/SKU/season combination, including Normal
# and any Above-Network Demand or Shortage rows that don't end up matched into an
# actual transfer (Section 10 only pairs up combinations where a suitable
# donor/recipient match exists).
status_report = profile.rename(columns={
    "magazakodu": "store", "urunkodu": "sku",
    "avg_demand": "avg_demand_units_per_day",
    "network_median_demand": "network_median_demand_units_per_day",
})[[
    "store", "sku", "season", "status",
    "avg_demand_units_per_day", "network_median_demand_units_per_day",
    "demand_vs_network", "stockout_rate_pct", "avg_daily_sales", "avg_daily_rev",
]].sort_values(["status", "store", "sku"])

status_report.to_csv(os.path.join(OUTPUT_DIR, "inventory_status_report.csv"), index=False)
print(f"Saved: {OUTPUT_DIR}/inventory_status_report.csv  "
      f"({len(status_report):,} rows -- every store/SKU/season combination, all statuses)")

excess   = profile[profile["status"] == "Above-Network Demand"]
shortage = profile[profile["status"].isin(["Critical Shortage", "Shortage Risk"])]

Inventory status classification (store x SKU x season):
status
Above-Network Demand    35273
Shortage Risk           24296
Critical Shortage       22857
Normal                   9399
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/inventory_status_report.csv  (91,825 rows -- every store/SKU/season combination, all statuses)


In [11]:
# =============================================================================
# PART 10 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 9,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 10. TRANSFER ALLOCATION
# =============================================================================
# Matches each shortage to the best available donor store(s), allocating
# available surplus once (no donor's stock is counted toward more than one
# destination), and rounds quantities up to full cartons.

allocated_rows = []
unused_capacity_rows = []

for (sku, season), grp_ in profile[profile["status"].isin(
        ["Critical Shortage", "Shortage Risk", "Above-Network Demand"])].groupby(["urunkodu", "season"], observed=True):

    donors = grp_[grp_["status"] == "Above-Network Demand"].copy()
    needs  = grp_[grp_["status"].isin(["Critical Shortage", "Shortage Risk"])].copy()
    if needs.empty:
        # No shortage anywhere for this SKU/season -- every donor's surplus goes unused.
        for _, d in donors.iterrows():
            surplus = max((d["avg_demand"] - d["network_median_demand"]) * (1 - DONOR_KEEP_BACK_RATIO) * TRANSFER_HORIZON_DAYS, 0)
            if surplus > 0:
                unused_capacity_rows.append({
                    "sku": sku, "season": season, "store": d["magazakodu"],
                    "unused_surplus_units": round(surplus, 2), "reason": "No shortage found for this SKU/season"
                })
        continue

    donors["remaining_surplus"] = (
        (donors["avg_demand"] - donors["network_median_demand"]) * (1 - DONOR_KEEP_BACK_RATIO) * TRANSFER_HORIZON_DAYS
    ).clip(lower=0)
    donors = donors[donors["remaining_surplus"] > 0].sort_values("remaining_surplus", ascending=False)

    needs["remaining_need"] = ((needs["network_median_demand"] - needs["avg_demand"]) * TRANSFER_HORIZON_DAYS).clip(lower=0)
    needs = needs.sort_values(["status", "demand_vs_network"], ascending=[True, True])

    donor_pool = donors.to_dict("records")
    for _, need_row in needs.iterrows():
        remaining_need = need_row["remaining_need"]
        matched_any = False

        if remaining_need > 0:
            for donor in donor_pool:
                if remaining_need <= 0:
                    break
                if donor["remaining_surplus"] <= 0 or donor["magazakodu"] == need_row["magazakodu"]:
                    continue
                allocate_units = min(donor["remaining_surplus"], remaining_need)
                if allocate_units <= 0:
                    continue
                transfer_cartons = math.ceil(allocate_units / UNITS_PER_CARTON)
                actual_units = min(transfer_cartons * UNITS_PER_CARTON, donor["remaining_surplus"])

                allocated_rows.append({
                    "sku": sku, "season": season,
                    "source_store": donor["magazakodu"], "destination_store": need_row["magazakodu"],
                    "status": need_row["status"], "match_found": True,
                    "transfer_cartons": transfer_cartons, "transfer_units": actual_units,
                    "destination_current_demand": round(need_row["avg_demand"], 2),
                    "network_median_demand": round(need_row["network_median_demand"], 2),
                    "source_current_demand": round(donor["avg_demand"], 2),
                })
                donor["remaining_surplus"] -= actual_units
                remaining_need              -= actual_units
                matched_any = True

        if not matched_any:
            # No donor was available (or none large enough) for this shortage --
            # still recorded, so it's visible rather than silently dropped.
            allocated_rows.append({
                "sku": sku, "season": season,
                "source_store": None, "destination_store": need_row["magazakodu"],
                "status": need_row["status"], "match_found": False,
                "transfer_cartons": 0, "transfer_units": 0,
                "destination_current_demand": round(need_row["avg_demand"], 2),
                "network_median_demand": round(need_row["network_median_demand"], 2),
                "source_current_demand": None,
            })

    # Any donor surplus not fully drawn down by the shortages above also goes unused.
    for donor in donor_pool:
        if donor["remaining_surplus"] > 0:
            unused_capacity_rows.append({
                "sku": sku, "season": season, "store": donor["magazakodu"],
                "unused_surplus_units": round(donor["remaining_surplus"], 2),
                "reason": "Surplus exceeded total shortage need for this SKU/season"
            })

allocated_df    = pd.DataFrame(allocated_rows)
unused_capacity_df = pd.DataFrame(unused_capacity_rows)

n_matched   = int(allocated_df["match_found"].sum()) if len(allocated_df) else 0
n_unmatched = len(allocated_df) - n_matched
print(f"Shortage cases reviewed: {len(allocated_df):,}")
print(f"  Matched to a donor store : {n_matched:,}")
print(f"  No donor available       : {n_unmatched:,}")
if n_matched:
    print(f"  Total units to be moved  : {allocated_df.loc[allocated_df['match_found'], 'transfer_units'].sum():,.0f}")
if len(unused_capacity_df):
    print(f"Above-Network Demand cases with unused donor capacity: {len(unused_capacity_df):,} "
          f"(total {unused_capacity_df['unused_surplus_units'].sum():,.0f} units sitting idle)")
else:
    print("Excess-stock cases with unused surplus: 0")

Shortage cases reviewed: 52,344
  Matched to a donor store : 41,029
  No donor available       : 11,315
  Total units to be moved  : 633,010
Above-Network Demand cases with unused donor capacity: 23,764 (total 1,065,775 units sitting idle)


In [12]:
# =============================================================================
# PART 11 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 10,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 11. TRANSFER URGENCY AND SCHEDULING
# =============================================================================

REFERENCE_DATE = df_full["tarih"].max()

trend_by_key = (
    df_full.groupby(["magazakodu", "urunkodu", "season"], observed=True)["ema_trend_u"]
    .mean().rename("destination_trend").reset_index()
    .rename(columns={"magazakodu": "destination_store", "urunkodu": "sku"})
)
allocated_df = allocated_df.merge(
    trend_by_key,
    on=["destination_store", "sku", "season"],
    how="left"
)
allocated_df["destination_trend"] = allocated_df["destination_trend"].fillna(0)

def urgency(row):
    if row["status"] == "Critical Shortage":
        return ("Immediate - 1 day", 1) if row["destination_trend"] < 0 else ("Immediate - 3 days", 3)
    if row["status"] == "Shortage Risk":
        return ("Urgent - 7 days", 7)
    return ("Planned - 14 days", 14)

allocated_df["urgency"], allocated_df["days_until_transfer"] = zip(*allocated_df.apply(urgency, axis=1))
allocated_df["preferred_transfer_date"] = REFERENCE_DATE + pd.to_timedelta(allocated_df["days_until_transfer"], unit="D")
allocated_df["preferred_transfer_date"] = allocated_df["preferred_transfer_date"].dt.date

print("Transfer urgency distribution:")
print(allocated_df["urgency"].value_counts().to_string())

Transfer urgency distribution:
urgency
Urgent - 7 days       28297
Immediate - 1 day     17828
Immediate - 3 days     6219


In [13]:
# =============================================================================
# PART 12 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 11,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 12. ESTIMATED VALUE IMPACT
# =============================================================================
# Unit values are estimated from historical revenue/units sold, since no
# fixed price list was available in the source data. Treat these figures as
# indicative estimates, not confirmed financial values.

sku_value_lookup = sku_value.set_index("urunkodu")["median_unit_value"].to_dict()

value_rows = []
for _, r in allocated_df.iterrows():
    unit_value = sku_value_lookup.get(r["sku"], np.nan)
    if pd.isna(unit_value) or unit_value <= 0:
        value_rows.append({"estimated_value_before": np.nan, "estimated_value_after": np.nan,
                            "estimated_value_uplift": np.nan, "uplift_pct": np.nan})
        continue

    value_before = r["destination_current_demand"] * TRANSFER_HORIZON_DAYS * unit_value
    restored_demand = max(
        r["destination_current_demand"],
        min(r["network_median_demand"], r["destination_current_demand"] + r["transfer_units"] / TRANSFER_HORIZON_DAYS)
    )
    value_after = restored_demand * TRANSFER_HORIZON_DAYS * unit_value
    uplift = value_after - value_before
    uplift_pct = (uplift / value_before * 100) if value_before > 0 else np.nan

    value_rows.append({"estimated_value_before": round(value_before, 2),
                        "estimated_value_after": round(value_after, 2),
                        "estimated_value_uplift": round(uplift, 2),
                        "uplift_pct": round(uplift_pct, 2) if pd.notna(uplift_pct) else np.nan})

value_detail = pd.DataFrame(value_rows)
allocated_df = pd.concat([allocated_df.reset_index(drop=True), value_detail], axis=1)

print(f"Total estimated value uplift: {allocated_df['estimated_value_uplift'].sum():,.2f}")
print(f"Median uplift per transfer: {allocated_df['uplift_pct'].median():.1f}%")
print("\nEstimated value uplift by status:")
print(allocated_df.groupby("status")["estimated_value_uplift"].sum().to_string())

Total estimated value uplift: 12,526,513.69
Median uplift per transfer: 155.8%

Estimated value uplift by status:
status
Critical Shortage    4329381.66
Shortage Risk        8197132.03


In [14]:
# =============================================================================
# PART 13 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 12,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 13. NETWORK-LEVEL IMPACT SUMMARY
# =============================================================================

ledger_rows = []
for _, r in allocated_df.iterrows():
    if not r["match_found"]:
        continue   # no transfer actually happened -- nothing to add to the ledger
    unit_value = sku_value_lookup.get(r["sku"], np.nan)
    if pd.isna(unit_value) or unit_value <= 0:
        continue

    src_before = r["source_current_demand"] * TRANSFER_HORIZON_DAYS
    src_after  = src_before - r["transfer_units"]

    dst_before    = r["destination_current_demand"] * TRANSFER_HORIZON_DAYS
    dst_after_raw = dst_before + r["transfer_units"]                                          # uncapped: full amount received
    dst_after_capped = min(dst_after_raw, r["network_median_demand"] * TRANSFER_HORIZON_DAYS)  # capped: conservative estimate

    src_value_delta = (src_after - src_before) * unit_value
    dst_value_delta_capped   = (dst_after_capped - dst_before) * unit_value
    dst_value_delta_uncapped = (dst_after_raw    - dst_before) * unit_value

    ledger_rows.append({"role": "source",      "value_change_conservative": src_value_delta,
                         "value_change_uncapped": src_value_delta})
    ledger_rows.append({"role": "destination",  "value_change_conservative": dst_value_delta_capped,
                         "value_change_uncapped": dst_value_delta_uncapped})

ledger_df = pd.DataFrame(ledger_rows)

# CONSERVATIVE: destination credited only up to the network median demand --
# never shows a receiving store as gaining more value than a normal store
# would generate. This is the cautious, deliberately understated figure.
#
# UNCAPPED: destination credited for the full value of stock received, matching
# how the source side is debited. This reflects the full value of stock moved,
# without the conservative cap.
#
# The two will differ whenever a shipment (rounded up to a full carton) is
# larger than the exact amount needed to bring a destination store up to the
# network median -- which is common, since transfers are sized in cartons.

net_value_conservative = ledger_df["value_change_conservative"].sum()
net_value_uncapped     = ledger_df["value_change_uncapped"].sum()

network_summary = pd.DataFrame([{
    "total_transfers": n_matched,
    "total_units_moved": allocated_df.loc[allocated_df["match_found"], "transfer_units"].sum(),
    "estimated_net_value_change_conservative": round(net_value_conservative, 2),
    "estimated_net_cost_change_conservative":  round(net_value_conservative * ASSUMED_COGS_RATIO, 2),
    "estimated_net_margin_change_conservative": round(net_value_conservative * (1 - ASSUMED_COGS_RATIO), 2),
    "estimated_net_value_change_uncapped": round(net_value_uncapped, 2),
    "estimated_net_cost_change_uncapped":  round(net_value_uncapped * ASSUMED_COGS_RATIO, 2),
    "estimated_net_margin_change_uncapped": round(net_value_uncapped * (1 - ASSUMED_COGS_RATIO), 2),
}])
network_summary.to_csv(os.path.join(OUTPUT_DIR, "network_impact_summary.csv"), index=False)

print("Network-level impact summary:")
print(f"  Conservative estimate (destination capped at network median demand):")
print(f"    Net value change  : {net_value_conservative:,.2f}")
print(f"  Uncapped estimate (destination credited for full stock received):")
print(f"    Net value change  : {net_value_uncapped:,.2f}")
print(f"Saved: {OUTPUT_DIR}/network_impact_summary.csv")

Network-level impact summary:
  Conservative estimate (destination capped at network median demand):
    Net value change  : -8,291,946.12
  Uncapped estimate (destination credited for full stock received):
    Net value change  : 0.00
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/network_impact_summary.csv


In [15]:
# =============================================================================
# PART 14 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 13,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 14. FINAL TRANSFER RECOMMENDATIONS — full detail, saved and displayed
# =============================================================================

FINAL_COLUMNS = [
    "sku", "season", "source_store", "destination_store", "status",
    "transfer_cartons", "transfer_units",
    "destination_current_demand", "network_median_demand",
    "urgency", "preferred_transfer_date",
    "estimated_value_before", "estimated_value_after", "estimated_value_uplift", "uplift_pct",
]

# Matched transfers -- an actual source store and date exist for these.
final_recommendations = allocated_df[allocated_df["match_found"]][FINAL_COLUMNS].sort_values(
    ["status", "estimated_value_uplift"], ascending=[True, False]
).reset_index(drop=True)
final_recommendations.to_csv(os.path.join(OUTPUT_DIR, "transfer_recommendations.csv"), index=False)

# Unmatched shortages -- flagged as needing shortage, but no donor store was
# available anywhere in the network for this SKU/season, so no source store
# or transfer date can be given. These need external replenishment, not an
# internal transfer.
unmatched_shortages = allocated_df[~allocated_df["match_found"]][
    ["sku", "season", "destination_store", "status", "destination_current_demand", "network_median_demand"]
].rename(columns={"destination_store": "store"}).sort_values(["status", "store"]).reset_index(drop=True)
unmatched_shortages.to_csv(os.path.join(OUTPUT_DIR, "unmatched_shortages.csv"), index=False)

# Unused donor capacity -- stores with above-network-demand surplus that was
# never allocated anywhere, either because no shortage existed for that
# SKU/season, or the surplus exceeded total network need.
unused_capacity_df.to_csv(os.path.join(OUTPUT_DIR, "unused_donor_capacity.csv"), index=False)

value_detail_export = allocated_df[allocated_df["match_found"]][
    ["sku", "source_store", "destination_store", "status",
     "estimated_value_before", "estimated_value_after", "estimated_value_uplift", "uplift_pct"]
]
value_detail_export.to_csv(os.path.join(OUTPUT_DIR, "value_impact.csv"), index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print(f"\nSaved: {OUTPUT_DIR}/transfer_recommendations.csv  ({len(final_recommendations):,} rows -- matched, with source store and date)")
print(f"Saved: {OUTPUT_DIR}/unmatched_shortages.csv          ({len(unmatched_shortages):,} rows -- no donor available)")
print(f"Saved: {OUTPUT_DIR}/unused_donor_capacity.csv        ({len(unused_capacity_df):,} rows -- surplus not allocated)")
print(f"Saved: {OUTPUT_DIR}/value_impact.csv")

print("\nTop 20 transfer recommendations by estimated value uplift:")
print(final_recommendations.head(20).to_string(index=False))

print("\nAll transfers due within the next 3 days:")
urgent_now = final_recommendations[final_recommendations["preferred_transfer_date"] <= (
    pd.Timestamp(REFERENCE_DATE) + pd.Timedelta(days=3)).date()]
print(urgent_now.to_string(index=False) if len(urgent_now) else "  None.")

if len(unmatched_shortages):
    print(f"\n{len(unmatched_shortages)} shortage case(s) have no available donor anywhere in the network "
          f"-- see unmatched_shortages.csv. These require external replenishment, not an internal transfer.")


Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/transfer_recommendations.csv  (41,029 rows -- matched, with source store and date)
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/unmatched_shortages.csv          (11,315 rows -- no donor available)
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/unused_donor_capacity.csv        (23,764 rows -- surplus not allocated)
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/value_impact.csv

Top 20 transfer recommendations by estimated value uplift:
        sku season  source_store destination_store            status  transfer_cartons  transfer_units  destination_current_demand  network_median_demand            urgency preferred_transfer_date  estimated_value_before  estimated_value_after  estimated_value_uplift  uplift_pct
 30000332.0 Summer        1016.0            5428.0 Critical Shortage                17           204.0                        3.13                  16.86 Immediate - 3 days              2026-01-06      

In [16]:
# =============================================================================
# PART 15 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 14,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 15. SUMMARY
# =============================================================================

print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print(f"  Forecasting model MAE          : {forecast_mae:.2f}")
print(f"  Reconciled forecast MAE        : {recon_mae:.2f}")
print(f"  Shortage cases matched to a transfer   : {n_matched:,}")
print(f"  Shortage cases with no donor available : {n_unmatched:,}")
print(f"  Above-Network Demand cases left unused  : {len(unused_capacity_df):,}")
print(f"  Total units to be moved        : {allocated_df.loc[allocated_df['match_found'], 'transfer_units'].sum():,.0f}")
print(f"  Net estimated value impact (conservative) : {network_summary['estimated_net_value_change_conservative'].iloc[0]:,.2f}")
print(f"  Net estimated value impact (uncapped)     : {network_summary['estimated_net_value_change_uncapped'].iloc[0]:,.2f}")
print(f"\n  All outputs saved to: {os.path.abspath(OUTPUT_DIR)}/")
print("    - transfer_recommendations.csv   (matched transfers: source store, cartons, date)")
print("    - unmatched_shortages.csv        (shortages with no donor found -- needs external stock)")
print("    - unused_donor_capacity.csv      (above-network-demand stores with nowhere to send surplus)")
print("    - inventory_status_report.csv    (full picture: every store/SKU/season, all statuses)")
print("    - value_impact.csv")
print("    - network_impact_summary.csv")
print("    - sku_categorisation.csv")
print("    - momentum_signals.csv")
print("    - backtest_results.csv           (see Section 16 below)")

PIPELINE COMPLETE
  Forecasting model MAE          : 120.15
  Reconciled forecast MAE        : 151.86
  Shortage cases matched to a transfer   : 41,029
  Shortage cases with no donor available : 11,315
  Above-Network Demand cases left unused  : 23,764
  Total units to be moved        : 633,010
  Net estimated value impact (conservative) : -8,291,946.12
  Net estimated value impact (uncapped)     : 0.00

  All outputs saved to: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/
    - transfer_recommendations.csv   (matched transfers: source store, cartons, date)
    - unmatched_shortages.csv        (shortages with no donor found -- needs external stock)
    - unused_donor_capacity.csv      (above-network-demand stores with nowhere to send surplus)
    - inventory_status_report.csv    (full picture: every store/SKU/season, all statuses)
    - value_impact.csv
    - network_impact_summary.csv
    - sku_categorisation.csv
    - momentum_signals.csv
    - backtest_results.csv           (see S

In [17]:
# =============================================================================
# PART 16 OF 16 -- Inventory Rebalancing Pipeline
# This is an extract from the master file (Inventory_Rebalancing_Pipeline.py)
# for readability only. It depends on variables created by Parts 1 through 15,
# and is not intended to be run standalone. Run the master file top to bottom,
# or paste these parts back together in order, to actually execute the pipeline.
# =============================================================================

# %%
# =============================================================================
# 16. BACKTEST -- did recommended transfers actually help?
# =============================================================================
# Re-runs status classification using ONLY data up to a historical cutoff
# (TRANSFER_HORIZON_DAYS before the end of the available data), so that real,
# already-observed sales data exists after the cutoff to check against. For
# each store/SKU that would have been flagged as needing a transfer at that
# cutoff, this compares actual average daily sales in the window BEFORE the
# cutoff (the store's own baseline) against actual average daily sales in the
# window AFTER the cutoff (the real, observed holdout period).
#
# This is a before/after comparison against each store's own trend, not a
# randomized controlled experiment -- there is no group of otherwise similar
# stores that did NOT receive a transfer to compare against, so this cannot
# establish causation. It can show whether recipients' sales moved in the
# expected direction after the hypothetical transfer point, which is the
# simplest check the available data supports.

print("\n" + "=" * 70)
print("SECTION 16 -- BACKTEST")
print("=" * 70)

backtest_cutoff  = df_full["tarih"].max() - pd.Timedelta(days=TRANSFER_HORIZON_DAYS)
holdout_end      = df_full["tarih"].max()
pre_window_start = backtest_cutoff - pd.Timedelta(days=TRANSFER_HORIZON_DAYS)

print(f"Backtest cutoff: {backtest_cutoff.date()} "
      f"(pre-window: {pre_window_start.date()} to {backtest_cutoff.date()}, "
      f"holdout window: {backtest_cutoff.date()} to {holdout_end.date()})")

df_backtest = df_full[df_full["tarih"] < backtest_cutoff]

if df_backtest["tarih"].nunique() < 60:
    print("Not enough historical data before the cutoff to run a meaningful backtest. Skipping.")
else:
    bt_profile = (
        df_backtest.groupby(["magazakodu", "urunkodu", "season"], observed=True)
        .agg(avg_demand=("ema_30_u", "mean"), days_recorded=("tarih", "count"),
             stockout_days=("stok_out", "sum"))
        .reset_index()
    )
    bt_profile["stockout_rate_pct"] = (bt_profile["stockout_days"] / bt_profile["days_recorded"] * 100).round(2)

    bt_benchmark = (
        df_backtest.groupby(["urunkodu", "season"], observed=True)["ema_30_u"]
        .median().reset_index().rename(columns={"ema_30_u": "network_median_demand"})
    )
    bt_profile = bt_profile.merge(bt_benchmark, on=["urunkodu", "season"], how="left")
    bt_profile["demand_vs_network"] = (
        bt_profile["avg_demand"] / bt_profile["network_median_demand"].replace(0, np.nan)
    ).round(3)
    bt_profile["status"] = bt_profile.apply(classify_status, axis=1)   # reuses Section 9's function

    bt_shortage    = bt_profile[bt_profile["status"].isin(["Critical Shortage", "Shortage Risk"])]
    bt_donors_all  = bt_profile[bt_profile["status"] == "Above-Network Demand"]

    bt_recipients = []
    for _, need_row in bt_shortage.iterrows():
        matches = bt_donors_all[
            (bt_donors_all["urunkodu"] == need_row["urunkodu"]) &
            (bt_donors_all["season"] == need_row["season"]) &
            (bt_donors_all["magazakodu"] != need_row["magazakodu"])
        ]
        if len(matches):
            bt_recipients.append({"store": need_row["magazakodu"], "sku": need_row["urunkodu"]})

    bt_recipients_df = pd.DataFrame(bt_recipients).drop_duplicates()
    print(f"Backtest: {len(bt_recipients_df):,} store/SKU combinations would have been flagged "
          f"for a transfer as of {backtest_cutoff.date()}.")

    if len(bt_recipients_df) == 0:
        print("No backtest recipients found -- nothing to validate in this window.")
    else:
        pre_sales = (
            df_full[(df_full["tarih"] >= pre_window_start) & (df_full["tarih"] < backtest_cutoff)]
            .groupby(["magazakodu", "urunkodu"], observed=True)["satismiktari"].mean()
            .rename("pre_avg_daily_sales")
        )
        pre_sales.index.names = ["store", "sku"]
        post_sales = (
            df_full[(df_full["tarih"] >= backtest_cutoff) & (df_full["tarih"] < holdout_end)]
            .groupby(["magazakodu", "urunkodu"], observed=True)["satismiktari"].mean()
            .rename("post_avg_daily_sales")
        )
        post_sales.index.names = ["store", "sku"]

        bt_recipients_df = bt_recipients_df.set_index(["store", "sku"])
        bt_recipients_df = bt_recipients_df.join(pre_sales, how="left").join(post_sales, how="left").dropna()
        bt_recipients_df["sales_change_pct"] = (
            (bt_recipients_df["post_avg_daily_sales"] - bt_recipients_df["pre_avg_daily_sales"])
            / bt_recipients_df["pre_avg_daily_sales"].replace(0, np.nan) * 100
        )

        n_total    = len(bt_recipients_df)
        n_improved = int((bt_recipients_df["post_avg_daily_sales"] > bt_recipients_df["pre_avg_daily_sales"]).sum())

        print(f"Recipients with both pre- and post-cutoff sales data: {n_total:,}")
        if n_total:
            print(f"Recipients whose sales INCREASED after the hypothetical transfer point: "
                  f"{n_improved:,} of {n_total:,} ({n_improved / n_total * 100:.1f}%)")
            print(f"Median sales change: {bt_recipients_df['sales_change_pct'].median():.1f}%")
            print("NOTE: before/after comparison against each store's own trend, not a controlled "
                  "experiment -- cannot establish causation on its own.")

        bt_recipients_df.reset_index().to_csv(os.path.join(OUTPUT_DIR, "backtest_results.csv"), index=False)
        print(f"Saved: {OUTPUT_DIR}/backtest_results.csv")


SECTION 16 -- BACKTEST
Backtest cutoff: 2025-12-20 (pre-window: 2025-12-06 to 2025-12-20, holdout window: 2025-12-20 to 2026-01-03)


Backtest: 14,571 store/SKU combinations would have been flagged for a transfer as of 2025-12-20.
Recipients with both pre- and post-cutoff sales data: 14,571
Recipients whose sales INCREASED after the hypothetical transfer point: 4,751 of 14,571 (32.6%)
Median sales change: -8.3%
NOTE: before/after comparison against each store's own trend, not a controlled experiment -- cannot establish causation on its own.
Saved: C:\Users\dpdea\Desktop\Cranfield\Thesis\output/backtest_results.csv
